# Diagnóstico post-hoc: clustering con encoder de dos etapas

Seleccionamos el checkpoint de menor error de cada seed sin filtrar por rango y medimos clustering en validation. El resultado es descriptivo: no cambia el FAIL formal ni autoriza test.

In [ ]:
# ruff: noqa: E402, E501
import json
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

from koopman_jepa.paper_config import (
    PaperCheckpointReplayConfig,
    load_paper_mlp_one_hidden_development_config,
)
from koopman_jepa.paper_data import PAPER_REGIME_NAMES, PaperRegimeDataset, generate_paper_master
from koopman_jepa.paper_evaluation import (
    evaluate_paper_mlp_clustering_gate,
    evaluate_paper_mlp_seed_clustering,
)
from koopman_jepa.paper_model import PaperTemporalJEPA
from koopman_jepa.paper_training import (
    make_paper_loader,
    run_paper_train_validation_with_checkpoint,
    summarize_seed_checkpoint,
    verify_paper_checkpoint_replay,
)

plt.style.use("seaborn-v0_8-whitegrid")
torch.use_deterministic_algorithms(True)

In [ ]:
config = load_paper_mlp_one_hidden_development_config(
    ROOT / "configs" / "paper_mlp_two_stage_one_hidden_development.yaml"
)
diagnostic_gate = replace(config.checkpoint_gate, min_validation_effective_rank=1.0)
replay_config = PaperCheckpointReplayConfig(metric_absolute_tolerance=1e-8)
train_dataset = PaperRegimeDataset(config.data, "train", generate_paper_master)
validation_dataset = PaperRegimeDataset(config.data, "val", generate_paper_master)
assert config.model.encoder_projection == "two_stage"
assert len(train_dataset) == 64 * len(PAPER_REGIME_NAMES) == 1152
assert len(validation_dataset) == 32 * len(PAPER_REGIME_NAMES) == 576
assert {train_dataset.sample_key(i) for i in range(len(train_dataset))}.isdisjoint(
    {validation_dataset.sample_key(i) for i in range(len(validation_dataset))}
)
print(f"Rango formal={config.checkpoint_gate.min_validation_effective_rank}; selección descriptiva={diagnostic_gate.min_validation_effective_rank}")
print("Test no fue instanciado.")

In [ ]:
models, summaries, times = {}, [], {}
for seed in config.sweep.seeds:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    run_config = replace(config.train, seed=seed)
    model = PaperTemporalJEPA(config.model)
    started = time.perf_counter()
    run = run_paper_train_validation_with_checkpoint(
        model, train_dataset, validation_dataset, run_config, diagnostic_gate
    )
    assert run.selection is not None
    replay = verify_paper_checkpoint_replay(
        model, run, validation_dataset, run_config, replay_config, expected_epoch=run.selection.epoch
    )
    assert replay.passed
    times[seed] = time.perf_counter() - started
    models[seed] = model
    summary = summarize_seed_checkpoint(seed, run.selection)
    summaries.append(summary)
    formal_rank = summary.validation_effective_rank >= config.checkpoint_gate.min_validation_effective_rank
    print(
        f"seed={seed} epoch={summary.checkpoint_epoch} val/base={summary.validation_loss_ratio:.3f} "
        f"rank={summary.validation_effective_rank:.2f} formal_rank={'PASS' if formal_rank else 'FAIL'}"
    )
print(f"Cinco checkpoints reproducidos; tiempo={sum(times.values()):.1f}s")

@torch.no_grad()
def collect_embeddings(model, run_config):
    device = torch.device(run_config.device)
    model.to(device).eval()
    embeddings, labels = [], []
    for context, _, batch_labels in make_paper_loader(validation_dataset, run_config, shuffle=False):
        embeddings.append(model.online_encoder(context.to(device)).cpu().numpy())
        labels.append(batch_labels.numpy())
    return np.concatenate(embeddings), np.concatenate(labels)

clustering_metrics, reference_labels = [], None
for seed in config.sweep.seeds:
    embeddings, labels = collect_embeddings(models[seed], replace(config.train, seed=seed))
    reference_labels = labels if reference_labels is None else reference_labels
    assert np.array_equal(labels, reference_labels)
    clustering_metrics.append(
        evaluate_paper_mlp_seed_clustering(embeddings, labels, seed, config.clustering)
    )
clustering_gate = evaluate_paper_mlp_clustering_gate(
    clustering_metrics, config.sweep, config.clustering_gate
)
print(json.dumps(asdict(clustering_gate), indent=2))

In [ ]:
summary_by_seed = {row.seed: row for row in summaries}
cluster_by_seed = {row.seed: row for row in clustering_metrics}
print("seed epoch rank formal_rank purity_mean purity_sd purity_min purity_max")
for seed in config.sweep.seeds:
    summary, cluster = summary_by_seed[seed], cluster_by_seed[seed]
    formal_rank = summary.validation_effective_rank >= config.checkpoint_gate.min_validation_effective_rank
    print(
        f"{seed:>4d} {summary.checkpoint_epoch:>5d} {summary.validation_effective_rank:>5.2f} "
        f"{str(formal_rank):>11s} {cluster.mean_purity:>11.2%} {cluster.purity_std:>9.2%} "
        f"{cluster.minimum_purity:>10.2%} {cluster.maximum_purity:>10.2%}"
    )

seeds = np.array(config.sweep.seeds)
means = np.array([cluster_by_seed[seed].mean_purity for seed in seeds])
stds = np.array([cluster_by_seed[seed].purity_std for seed in seeds])
ranks = np.array([summary_by_seed[seed].validation_effective_rank for seed in seeds])
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
axes[0].errorbar(seeds, means, yerr=stds, fmt="o", capsize=5)
axes[0].axhline(config.clustering_gate.min_worst_seed_mean_purity, color="tab:red", linestyle="--", label="mínimo seed")
axes[0].axhline(0.6548, color="tab:green", linestyle=":", label="paper")
axes[0].set(title="Pureza media ± sd", xlabel="Seed", ylabel="Pureza")
axes[0].legend()
axes[1].boxplot([cluster_by_seed[seed].purities for seed in seeds], tick_labels=seeds)
axes[1].axhline(config.clustering_gate.min_overall_mean_purity, color="tab:red", linestyle="--")
axes[1].set(title="Variación K-means", xlabel="Seed", ylabel="Pureza")
axes[2].scatter(ranks, means)
for seed, rank, mean in zip(seeds, ranks, means, strict=True):
    axes[2].annotate(str(seed), (rank, mean), xytext=(5, 5), textcoords="offset points")
axes[2].axvline(config.checkpoint_gate.min_validation_effective_rank, color="tab:red", linestyle="--")
axes[2].set(title="Rango vs pureza", xlabel="Rango efectivo", ylabel="Pureza")
plt.show()

status = "PASS descriptivo" if clustering_gate.passed else "FAIL descriptivo"
display(Markdown(f"""## Resultado descriptivo

- **{status}.**
- Pureza global: **{clustering_gate.overall_mean_purity:.2%}**.
- Peor seed: **{clustering_gate.worst_seed_mean_purity:.2%}**.
- CV entre seeds: **{clustering_gate.seed_mean_purity_coefficient_of_variation:.3f}**.
- Mayor sd de K-means: **{clustering_gate.worst_within_seed_purity_std:.2%}**.

Este resultado no modifica el FAIL formal y test no fue construido.
"""))